# Fraud Detection on the IEEE-CIS Dataset: Decision Tree vs LightGBM

**Goal:** predict whether an e-commerce transaction is fraudulent (`isFraud`), and compare a simple
baseline (Decision Tree) against a gradient-boosting model (LightGBM) using a **time-ordered
validation scheme** that avoids data leakage.

**Pipeline**
1. Load the CSV files
2. Look at the data
3. Explore promising features
4. Clean the data and create a few simple features
5. Preprocess: 6 monthly folds, ordered oldest to newest, with preprocessing fitted on training data only
6. Train and compare Decision Tree vs LightGBM
7. Interpret the results

## 0. Dataset quality and justification

**Why this dataset?**
- It contains **real** card-not-present e-commerce transactions provided by Vesta Corporation, so it is
  much closer to a real fraud problem than a synthetic or heavily pre-processed dataset.
- It is **large** (about 590k labelled transactions) with a mix of numeric and categorical features.
- It is **highly imbalanced** (about 3.5% fraud), which is the central difficulty of fraud detection and forces
  us to choose metrics carefully (accuracy would be meaningless).
- It has a **time dimension** (`TransactionDT`), so we can mimic deployment: train on the past, predict the future.
- It is a well-known public benchmark, so results can be put in context.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import OrdinalEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score, average_precision_score
import lightgbm as lgb

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 150)

DATA_DIR = Path("data")      # folder containing the Kaggle csv files
SEED = 42
SAMPLE_FRAC = 1.0            # set to e.g. 0.2 for a quick test run on a random 20% of the rows

## 1. Load the data

We load `train_transaction.csv` and `train_identity.csv` and left-join them on `TransactionID`
(every transaction is kept; identity info is attached where it exists).
To save memory we convert 64-bit numbers to 32-bit.

In [ ]:
trans = pd.read_csv(f"{DATA_DIR}/train_transaction.csv")
ident = pd.read_csv(f"{DATA_DIR}/train_identity.csv")

# safety: some Kaggle files write id-01 instead of id_01
ident.columns = ident.columns.str.replace("-", "_")
ident["has_identity"] = 1

df = trans.merge(ident, on="TransactionID", how="left")
df["has_identity"] = df["has_identity"].fillna(0).astype("int8")
del trans, ident

for c in df.select_dtypes("float64").columns:
    df[c] = df[c].astype("float32")
for c in df.select_dtypes("int64").columns:
    df[c] = df[c].astype("int32")

if SAMPLE_FRAC < 1:
    df = df.sample(frac=SAMPLE_FRAC, random_state=SEED).reset_index(drop=True)

print("Shape:", df.shape)

## 2. Look at the data
Basic sanity checks: size, types, target balance, missing values, duplicates.

In [ ]:
print("Rows, columns:", df.shape)
print("Column types:\n", df.dtypes.value_counts(), "\n")
print(f"Memory: {df.memory_usage(deep=True).sum() / 1e9:.2f} GB\n")

df[["TransactionID", "isFraud", "TransactionDT", "TransactionAmt", "ProductCD",
    "card4", "card6", "P_emaildomain", "DeviceType"]].head()

In [ ]:
fraud_rate = df["isFraud"].mean()
print(f"Fraud rate: {fraud_rate:.2%}")

In [ ]:
miss = df.isna().mean().sort_values(ascending=False)
print("Top 10 most-missing columns:\n", miss.head(10).round(3), "\n")
print("Columns with >90% missing:", int((miss > 0.9).sum()))
print("Columns with no missing   :", int((miss == 0).sum()))

print("\nDuplicate TransactionIDs:", df["TransactionID"].duplicated().sum())

In [ ]:
print("transactions with identity info:", f"{df['has_identity'].mean():.1%}")

## 3. Explore promising features
Quick univariate looks at fraud rate against a few intuitive features.
**Important:** this exploration is only used to *motivate* the simple engineered features in section 4.
We do **not** use it to select or drop columns (selection using labels from the whole dataset would leak
information from the future into the validation folds).

In [ ]:
def fraud_rate_bar(col, ax, top=8, min_n=500):
    s = df.groupby(col, dropna=False)["isFraud"].agg(["mean", "size"])
    s = s[s["size"] >= min_n].sort_values("mean", ascending=False).head(top)
    ax.bar([str(i) for i in s.index], s["mean"] * 100)
    ax.set_title(f"Fraud rate (%) by {col}")
    ax.tick_params(axis="x", rotation=45)

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for ax, col in zip(axes.ravel(), ["ProductCD", "card4", "card6", "DeviceType", "P_emaildomain", "has_identity"]):
    fraud_rate_bar(col, ax)
plt.tight_layout(); plt.show()

## 4. Cleaning and simple feature engineering

Everything in this step is **row-wise** (each new value depends only on that row), so it cannot leak
information between train and test. Anything that needs statistics from the data (missing-rate
thresholds, category encodings) is deferred to section 5 and fitted on training data only.

Steps:
- Lower-case and strip all text columns (`"Visa"` and `"visa "` become the same value).
- `hour`, `day_of_week` from `TransactionDT`.
- `amt_log`, `amt_cents` from `TransactionAmt`.
- `email_match`: purchaser and recipient email domain are the same (1/0, NaN if one is missing).
- `browser_family`, `os_family`: strip version numbers from `id_31` / `id_30`
  (`"chrome 63.0"` becomes `"chrome"`).
- `n_missing`: number of missing values in the row (a cheap way to capture "how much info do we have").
- Drop `TransactionID` (just a key) and `TransactionDT` as a *feature*. We keep it only to build months.
  A raw time counter would never repeat in the future, so a model could not use it meaningfully.

In [ ]:
def clean_and_engineer(d):
    # 1) text columns -> lower-case, stripped
    text_cols = [c for c in d.columns]
    for c in text_cols:
        d[c] = d[c].astype("object").where(d[c].isna(), d[c].astype(str).str.lower().str.strip())

    # 2) time features
    d["hour"] = ((d["TransactionDT"] // 3600) % 24).astype("int8")
    d["day_of_week"] = ((d["TransactionDT"] // 86400) % 7).astype("int8")

    # 3) amount features
    d["amt_log"] = np.log1p(d["TransactionAmt"]).astype("float32")
    d["amt_cents"] = ((d["TransactionAmt"] * 100).round() % 100).astype("float32")

    # 4) email match
    both = d["P_emaildomain"].notna() & d["R_emaildomain"].notna()
    d["email_match"] = np.where(both, (d["P_emaildomain"] == d["R_emaildomain"]).astype(float), np.nan).astype("float32")

    # 5) browser / OS family (remove version numbers)
    for src, dst in [("id_31", "browser_family"), ("id_30", "os_family")]:
        if src in d.columns:
            d[dst] = d[src].str.replace(r"[\d\._]+", "", regex=True).str.strip()

    # 6) number of missing values in the row (counted on original columns only)
    d["n_missing"] = d.isna().sum(axis=1).astype("int16")
    return d

df = clean_and_engineer(df)

# --- time folds: 6 blocks of 30 days, oldest = 0 ... newest = 5 (anything beyond day 180 goes to the last block)
df["month"] = (df["TransactionDT"] // (30 * 86400)).clip(upper=5).astype(int)

# --- split features / target / month
y = df["isFraud"].values
month = df["month"].values
X = df.drop(columns=["isFraud", "TransactionID", "TransactionDT", "month"])
del df


## 5. Preprocessing and the 6 monthly folds

**Folds.** The data is cut into 6 blocks of 30 days (fold 0 = oldest month, fold 5 = newest).
We use **forward chaining (expanding window)**:

| Round | Train on | Test on |
|---|---|---|
| 1 | month 0 | month 1 |
| 2 | months 0–1 | month 2 |
| 3 | months 0–2 | month 3 |
| 4 | months 0–3 | month 4 |
| 5 | months 0–4 | month 5 |

The model is never tested on data from before its training data, so there is no look-ahead.
Month 0 can only be used for training, which is why there are 5 test rounds rather than 6.

**No-leakage rules.** A `Preprocessor` object is **fitted on the training months only** and then applied to the
test month. It learns:
1. which columns to drop (more than 90% missing in train, or constant in train);
2. the integer code of each category (categories unseen in train become `-1`; missing becomes its own category).

Nothing from the test month (not even its labels or its missing rates) is used to build the preprocessing.
For the Decision Tree, missing numeric values are filled with a constant (`-999`), which needs no statistics. LightGBM handles
missing values natively, so it receives the NaNs as they are. Both models see exactly the same columns.

In [ ]:
class Preprocessor:
    """Fit on the training months only, then transform any month."""

    def __init__(self, max_missing=0.9):
        self.max_missing = max_missing

    def fit(self, X):
        miss = X.isna().mean()
        self.keep_ = [c for c in X.columns
                      if miss[c] <= self.max_missing and X[c].nunique(dropna=True) > 1]
        self.cat_cols_ = [c for c in self.keep_ if not pd.api.types.is_numeric_dtype(X[c])]
        self.num_cols_ = [c for c in self.keep_ if c not in self.cat_cols_]
        self.enc_ = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
        self.enc_.fit(X[self.cat_cols_].astype(object).fillna("missing"))
        return self

    def transform(self, X):
        cat = pd.DataFrame(
            self.enc_.transform(X[self.cat_cols_].astype(object).fillna("missing")),
            columns=self.cat_cols_, index=X.index)
        out = pd.concat([X[self.num_cols_], cat], axis=1)
        return out.astype("float32")


# overview of the folds
fold_info = pd.DataFrame({"month": month, "isFraud": y}).groupby("month")["isFraud"].agg(n_rows="size", n_fraud="sum", fraud_rate="mean")
fold_info["fraud_rate"] = (fold_info["fraud_rate"] * 100).round(2)
fold_info

## 6. Models and evaluation

**Models**
- **Decision Tree (baseline):** simple, fast and interpretable. `max_depth=8`, `min_samples_leaf=100` to prevent
  overfitting (a fully grown tree memorises the training data).
- **LightGBM:** gradient-boosted trees, usually the strongest choice for tabular data. Handles missing values
  and many features efficiently. Fixed settings (300 trees, learning rate 0.05, 64 leaves, column/row
  subsampling). **No early stopping** on the test month, since that would leak test information into training.
  No hyper-parameter tuning is done, to keep the project simple.

We do **not** re-balance the classes (no oversampling or class weights). The chosen metrics are
threshold-free or rank-based, so this keeps the predicted probabilities honest.

**Metrics** (because only about 3.5% of transactions are fraud):
- **ROC-AUC**: probability that a random fraud is scored higher than a random legitimate transaction (the Kaggle metric).
  It can look flattering under heavy imbalance.
- **PR-AUC (average precision)**: summarises the precision/recall trade-off on the fraud class. A random model scores
  about the fraud rate (about 0.035), so this metric is much more demanding.
- **Recall@1% and Precision@1%**: imagine analysts can review only the 1% highest-scored transactions.
  Recall@1% = share of all frauds caught in that 1%. Precision@1% = share of reviewed transactions that are really fraud.
  This is a practical, threshold-free way to measure business value.

In [ ]:
def evaluate(y_true, score, top_frac=0.01):
    n_top = max(1, int(len(y_true) * top_frac))
    top = np.argsort(-score, kind="stable")[:n_top]
    return {
        "roc_auc": roc_auc_score(y_true, score),
        "pr_auc": average_precision_score(y_true, score),
        "recall@1%": y_true[top].sum() / y_true.sum(),

    }

def make_models():
    return {
        "Decision Tree": DecisionTreeClassifier(max_depth=8,
                                                min_samples_leaf=100,
                                                random_state=SEED),
        "LightGBM": lgb.LGBMClassifier(
            n_estimators=300,
            learning_rate=0.05,
            num_leaves=64,
            min_child_samples=50,
            subsample=0.8,
            subsample_freq=1,
            colsample_bytree=0.5,
            random_state=SEED,
            n_jobs=-1,
            verbose=-1),
    }

rows, last_fold = [], {}
for k in range(1, 6):                              # test months 1..5
    tr, te = month < k, month == k                 # train = all earlier months
    prep = Preprocessor().fit(X[tr])               # <- fitted on TRAIN only
    X_tr, X_te = prep.transform(X[tr]), prep.transform(X[te])
    y_tr, y_te = y[tr], y[te]

    for name, model in make_models().items():
        if name == "Decision Tree":                # trees in sklearn need a constant fill for NaN
            model.fit(X_tr.fillna(-999), y_tr)
            score = model.predict_proba(X_te.fillna(-999))[:, 1]
        else:
            model.fit(X_tr, y_tr)
            score = model.predict_proba(X_te)[:, 1]
        rows.append({"test_month": k, "model": name, "n_train": int(tr.sum()), "n_test": int(te.sum()),
                     "fraud_rate_test": y_te.mean(), **evaluate(y_te, score)})
        last_fold[name] = (model, list(X_tr.columns))
    print(f"fold {k} done  (train rows={tr.sum():,}, features kept={X_tr.shape[1]})")

results = pd.DataFrame(rows)
results.round(4)

In [ ]:
metrics = ["roc_auc", "pr_auc", "recall@1%"]
summary = results.groupby("model")[metrics].agg(["mean", "std"]).round(4)
display(summary)

# per-fold comparison plot
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, m in zip(axes, metrics):
    for name, g in results.groupby("model"):
        ax.plot(g["test_month"], g[m], marker="o", label=name)
    ax.set_title(m); ax.set_xlabel("test month"); ax.set_xticks(range(1, 6))
axes[0].legend()
plt.tight_layout(); plt.show()

wins = results.pivot(index="test_month", columns="model", values="pr_auc")
print("LightGBM beats the Decision Tree (PR-AUC) in", int((wins["LightGBM"] > wins["Decision Tree"]).sum()), "of", len(wins), "folds")

### Which features matter?
Feature importance from the models trained on the **last round** (months 0–4). Importance shows what the model *uses*,
not causal proof of what causes fraud.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for ax, name in zip(axes, ["Decision Tree", "LightGBM"]):
    model, cols = last_fold[name]
    imp = (model.booster_.feature_importance("gain") if name == "LightGBM" else model.feature_importances_)
    s = pd.Series(imp, index=cols).sort_values(ascending=False).head(15)[::-1]
    ax.barh(s.index, s.values / s.values.sum())
    ax.set_title(f"{name}: top 15 features (share of importance)")
plt.tight_layout(); plt.show()

## 7. Interpretation of results

**How to read the numbers**
- Compare models **fold by fold** and look at the **mean and standard deviation** across folds. A gap that is larger than the fold-to-fold
  variation is more convincing than a small gap in a single fold.
- ROC-AUC will look high for both models, since fraud is rare and many legitimate transactions are easy to rank low.
  Rely more on **PR-AUC** and **Recall@1%**, which reflect how well the model finds the rare fraud class.
- Remember the reference point: a random model has PR-AUC ≈ fraud rate (about 0.035).

**What we expect (to be confirmed or rejected by your run):** LightGBM should outperform the single tree on every
metric, because boosting combines many trees and captures interactions between the hundreds of anonymised
features, whereas one shallow tree uses only a handful of splits. The tree is still a useful baseline since it is
fast and easy to explain.

**Discussion points to write up after running**
- Did LightGBM win in every fold? By how much in PR-AUC and Recall@1%?
- Do results change between folds (e.g. later months being easier or harder)? This reflects drift in fraud patterns.
- Which features were most important in each model? Are they the ones identified in section 3?

**Limitations**
- No hyper-parameter tuning, so both models could be improved (especially LightGBM).
- Most features are anonymised, so we cannot give business explanations for them.
- Months are approximate 30-day blocks. Only 5 test rounds, so uncertainty estimates are rough.
- We did not use powerful but more complex ideas from the Kaggle competition (client "UID" reconstruction, aggregations on card/address groups).
  Those would improve scores but would need careful leakage control, and they are outside the scope of this project.
- The test set of the original competition is unlabelled, so our scores are **not** comparable with Kaggle leaderboard scores.